In [23]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import random
from EloTracker import EloTracker

print("Import successful - no need to install ")

Import successful - no need to install 


data prep

In [13]:
#import the data
data = "/Users/dtruongs/Documents/GitHub/world cup 2026 prediction/international_football/results.csv"

df = pd.read_csv(data)

#df.head(10)
df.tail(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49368,2026-06-26,Cape Verde,Saudi Arabia,NaN,NaN,FIFA World Cup,Houston,United States,True
49369,2026-06-26,Uruguay,Spain,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
49370,2026-06-26,Norway,France,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49371,2026-06-26,Senegal,Iraq,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49372,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49373,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49374,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49375,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49376,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49377,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


In [14]:
dict_correct = {
    "Cape Verde": "Cabo Verde",
    "Czech Republic":"Czechia",
    "Ireland":"Republic of Ireland",
    "DR Congo": "Congo DR",
    "United States":"USA",
    "Iran": "IR Iran",
    "South Korea":"Korea Republic",
    "Ivory Coast": "Côte d'Ivoire",
    "Turkey": "Türkiye",
    "Gambia": "The Gambia"
}

df['home_team'] = df['home_team'].replace(dict_correct)
df['away_team'] = df['away_team'].replace(dict_correct)

date analyzing

In [15]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
matchs = len(df)
most_old = min(df['date'])
most_recent = max(df['date'])
print(matchs)
print(most_old)
print(most_recent)

49378
1872-11-30 00:00:00
2026-06-27 00:00:00


In [25]:
df['date'] = pd.to_datetime(df['date'])
df_modern = df[df['date'].dt.year >= 2000].sort_values('date').copy()
df_modern.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24062,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False
24063,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False
24064,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False
24065,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False
24066,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True
24067,2000-01-09,Côte d'Ivoire,Egypt,2.0,0.0,Friendly,Abidjan,Ivory Coast,False
24068,2000-01-09,Mexico,IR Iran,2.0,1.0,Friendly,Oakland,United States,True
24069,2000-01-11,Bermuda,Canada,0.0,2.0,Friendly,Hamilton,Bermuda,False
24070,2000-01-11,Burkina Faso,Cameroon,2.0,2.0,Friendly,Ouagadougou,Burkina Faso,False
24071,2000-01-13,Senegal,Cameroon,0.0,0.0,Friendly,Dakar,Senegal,False


In [26]:
conditions = [
    df['home_score'] > df['away_score'],
    df['home_score'] == df['away_score'],
    df['home_score'] < df['away_score']
]
choices = [1.0, 0.5, 0.0]
df['home_elo_result'] = np.select(conditions, choices, default=np.nan)

In [29]:
#set up elo tracker 
tracker = EloTracker(k_factor=30)
home_elo = []
away_elo = []

for index, row in df.iterrows():
    if pd.isna(row['home_elo_result']):
        #initialize the ratings for new teams
        home_elo.append(tracker.get_rating(row['home_team']))
        away_elo.append(tracker.get_rating(row['away_team']))

    else: 
        pre_match_home, pre_match_away = tracker.update_ratings(row["home_team"], row["away_team"], row["home_elo_result"])
        home_elo.append(pre_match_home)
        away_elo.append(pre_match_away)

In [30]:
#append to the df 
df['home_elo_pre_match'] = home_elo
df['away_elo_pre_match'] = away_elo
df["del_elo"] = df['home_elo_pre_match'] - df['away_elo_pre_match']

print(df.head(10))

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3 1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4 1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   
5 1876-03-25  Scotland     Wales         4.0         0.0   Friendly  Glasgow   
6 1877-03-03   England  Scotland         1.0         3.0   Friendly   London   
7 1877-03-05     Wales  Scotland         0.0         2.0   Friendly  Wrexham   
8 1878-03-02  Scotland   England         7.0         2.0   Friendly  Glasgow   
9 1878-03-23  Scotland     Wales         9.0         0.0   Friendly  Glasgow   

    country  neutral  home_elo_result  home_elo_pre_match  away_elo_pre_match  \
0  Scotland    False              0.5 